In [ ]:
import pandas as pd
import numpy as np
import pyodbc
import os
import time
from datetime import datetime

def get_sql_connection():
    """ایجاد اتصال به SQL Server"""
    return pyodbc.connect(
        'DRIVER={SQL Server};'
        'SERVER=MKZ-DSAS\\DSAS;'
        'DATABASE=DSAS;'
        'UID=datadriven;'
        'PWD=5Rdx@4Rfv1362'
    )

def fetch_and_transform_data():
    """
    خواندن داده از SQL Server، تغییر استراکچر و ذخیره در فایل اکسل
    """
    
    print("="*60)
    print(f"🔄 شروع اجرا در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)
    
    # مرحله 1: خواندن داده از SQL Server
    print("📥 مرحله 1: خواندن داده از SQL Server...")
    
    query = """
    SELECT TOP (10000000) [ID]
          ,[AssetID]
          ,[UnitID]
          ,[Value]
          ,[RecordTime]
          ,[RecordDate]
          ,[PersonelID]
          ,[OutofRange]
          ,[ValueType]
          ,[MobileID]
          ,[DateTime]
          ,[TimeStamp]
          ,[Job]
          ,[IsDeleted]
          ,[ShiftCode]
          ,[OnTime]
    FROM [DSAS].[PDA].[Periodic_Values]
    WHERE [UnitID]=11 AND
    ('AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
     'AssetID_9368', 'AssetID_9369', 'AssetID_9370',
     'AssetID_9343', 'AssetID_9344', 'AssetID_9408')
    """
    
    try:
        conn = get_sql_connection()
        df = pd.read_sql(query, conn)
        conn.close()
        print(f"✅ داده با موفقیت از SQL خوانده شد. تعداد رکوردها: {len(df):,}")
    except Exception as e:
        print(f"❌ خطا در خواندن داده از SQL: {e}")
        return None
    
    # مرحله 2: تغییر استراکچر داده (همانند کد اول)
    print("🔄 مرحله 2: تغییر استراکچر داده...")
    
    # استخراج AssetIDهای یکتا
    unique_ids = df['AssetID'].unique().tolist()
    print(f"🔢 تعداد AssetIDهای یکتا: {len(unique_ids)}")
    
    # ساخت دیتافریم خروجی با ستون‌های مورد نظر
    columns = ['TimeStamp'] + [f'AssetID_{uid}' for uid in unique_ids]
    output_df = pd.DataFrame(columns=columns)
    
    # پردازش تکراری تا خالی شدن دیتافریم اصلی
    row_count = 0
    while not df.empty and unique_ids:
        main_id = unique_ids[0]
        main_subset = df[df['AssetID'] == main_id]
        
        if main_subset.empty:
            unique_ids.pop(0)
            continue
        
        # گرفتن اولین ردیف از AssetID اصلی
        main_row = main_subset.iloc[0]
        main_ts = main_row['TimeStamp']
        main_value = main_row['Value']
        used_indices = [main_row.name]
        
        # پیدا کردن نزدیک‌ترین TimeStamp برای سایر AssetIDها
        row_data = {'TimeStamp': main_ts, f'AssetID_{main_id}': main_value}
        
        for other_id in unique_ids[1:]:
            subset = df[df['AssetID'] == other_id].copy()
            if subset.empty:
                row_data[f'AssetID_{other_id}'] = np.nan
                continue
            
            subset['ts_diff'] = np.abs(subset['TimeStamp'] - main_ts)
            close_rows = subset[subset['ts_diff'] <= 1800]
            
            if not close_rows.empty:
                closest_row = close_rows.sort_values('ts_diff').iloc[0]
                row_data[f'AssetID_{other_id}'] = closest_row['Value']
                used_indices.append(closest_row.name)
            else:
                row_data[f'AssetID_{other_id}'] = np.nan
        
        # حذف ردیف‌های استفاده‌شده
        df.drop(index=used_indices, inplace=True)
        
        # اضافه کردن ردیف جدید به خروجی
        output_df = pd.concat([output_df, pd.DataFrame([row_data])], ignore_index=True)
        row_count += 1
        
        # نمایش پیشرفت
        if row_count % 1000 == 0:
            print(f"   پردازش {row_count:,} رکورد...")
    
    print(f"✅ تعداد رکوردهای پردازش شده: {row_count:,}")
    
    # مرحله 3: حذف ردیف‌های دارای NaN
    print("🧹 مرحله 3: حذف ردیف‌های دارای مقادیر خالی...")
    asset_columns = [col for col in output_df.columns if col.startswith('AssetID_')]
    before_count = len(output_df)
    output_df = output_df.dropna(subset=asset_columns, how='any')
    after_count = len(output_df)
    print(f"   حذف {before_count - after_count:,} ردیف دارای مقادیر خالی")
    
    # مرحله 4: تبدیل TimeStamp به تاریخ و زمان
    print("📅 مرحله 4: تبدیل زمان‌ها...")
    output_df['date'] = pd.to_datetime(output_df['TimeStamp'], unit='s')
    output_df['RecordDate'] = output_df['date'].dt.date
    output_df['RecordTime'] = output_df['date'].dt.time
    
    # اضافه کردن ستون id
    output_df.insert(0, 'id', range(1, len(output_df) + 1))
    
    # حذف ستون TimeStamp
    output_df.drop(columns=['TimeStamp'], inplace=True)
    
    # مرتب‌سازی بر اساس تاریخ
    output_df.sort_values(by='date', inplace=True)
    
    # مرحله 5: ذخیره در فایل اکسل با نام ثابت
    print("💾 مرحله 5: ذخیره در فایل اکسل...")
    
    # تعیین مسیر خروجی
    output_dir = r'second_stage_inputs\G11'
    
    # ایجاد پوشه در صورت عدم وجود
    os.makedirs(output_dir, exist_ok=True)
    
    # نام ثابت فایل (بدون تغییر)
    output_file = os.path.join(output_dir, 'dsas_g11_turbine_bearings_output.xlsx')
    
    try:
        # ذخیره مستقیم با نام ثابت
        output_df.to_excel(output_file, index=False)
        print(f"✅ فایل با موفقیت ذخیره شد: {output_file}")
        print(f"📊 تعداد رکوردهای نهایی: {len(output_df):,}")
        print(f"📋 تعداد ستون‌ها: {len(output_df.columns)}")
        
        # ذخیره یک نسخه پشتیبان با تاریخ (اختیاری)
        backup_dir = os.path.join(output_dir, 'backup')
        os.makedirs(backup_dir, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M")
        backup_file = os.path.join(backup_dir, f'dsas_g11_generator_bearings_output_{timestamp}.xlsx')
        output_df.to_excel(backup_file, index=False)
        print(f"✅ نسخه پشتیبان ذخیره شد: {backup_file}")
        
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل: {e}")
        return None
    
    # نمایش اطلاعات آماری
    print("\n📊 اطلاعات آماری:")
    print(f"   بازه تاریخ: {output_df['date'].min()} تا {output_df['date'].max()}")
    print(f"   تعداد AssetIDها: {len(asset_columns)}")
    print("\n📋 نمونه‌ای از داده‌ها:")
    print(output_df.head(10))
    
    print("="*60)
    print(f"✅ فرآیند در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} کامل شد")
    print("="*60)
    
    return output_df

def run_scheduler():
    """
    بررسی مداوم برای اجرا در زمان‌های مشخص (هر روز)
    """
    print("="*60)
    print("🔄 برنامه زمان‌بندی خودکار شروع به کار کرد")
    print("="*60)
    print("⏰ زمان‌های اجرا (هر روز):")
    print("   - ساعت 09:00")
    print("   - ساعت 21:00")
    print("="*60)
    print("💡 برای توقف برنامه، Ctrl+C را بزنید")
    print("="*60)
    
    last_run_time = None  # فقط برای جلوگیری از اجرای مجدد در یک زمان
    
    while True:
        try:
            now = datetime.now()
            current_time = now.strftime("%H:%M")
            
            # بررسی زمان‌های مشخص
            if current_time in ["11:45", "21:00"]:
                # فقط چک می‌کنیم که در همین زمان دوبار اجرا نشود
                if last_run_time != current_time:
                    print("\n" + "="*60)
                    print(f"⏰ زمان اجرا فرا رسید: {now.strftime('%Y-%m-%d %H:%M:%S')}")
                    print("="*60)
                    
                    # اجرای تابع اصلی
                    result_df = fetch_and_transform_data()
                    
                    if result_df is not None:
                        print("\n" + "="*60)
                        print("✅ اجرای زمان‌بندی شده با موفقیت کامل شد!")
                        print("="*60)
                    else:
                        print("\n" + "="*60)
                        print("❌ اجرای زمان‌بندی شده با شکست مواجه شد!")
                        print("="*60)
                    
                    # ثبت زمان اجرا
                    last_run_time = current_time
                    
                    # 10 ثانیه صبر کن تا از اجرای مجدد در همان دقیقه جلوگیری شود
                    time.sleep(10)
            
            # هر 10 ثانیه یکبار بررسی کن
            time.sleep(10)
            
        except KeyboardInterrupt:
            print("\n" + "="*60)
            print("⏹️ برنامه با دستور کاربر متوقف شد")
            print(f"⏹️ زمان توقف: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("="*60)
            break
            
        except Exception as e:
            print(f"❌ خطا در حلقه اصلی: {e}")
            print("🔄 ادامه اجرا...")
            time.sleep(60)

# اجرای اصلی
if __name__ == "__main__":
    try:
        print("="*60)
        print("🚀 شروع برنامه دریافت داده از SQL")
        print("="*60)
        
        # شروع زمان‌بندی
        run_scheduler()
        
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        input("برای خروج Enter بزنید...")

🚀 شروع برنامه دریافت داده از SQL
🔄 برنامه زمان‌بندی خودکار شروع به کار کرد
⏰ زمان‌های اجرا (هر روز):
   - ساعت 09:00
   - ساعت 21:00
💡 برای توقف برنامه، Ctrl+C را بزنید
